In [33]:
import os
import time
from pathlib import Path

import numpy as np
import pandas as pd
import requests

os.environ["CENSUS_API_KEY"] = "b1278b466e026ad679a62a03dd4b044bb8b36e7c"
CENSUS_API_KEY = os.getenv("CENSUS_API_KEY")
print("Key loaded:", CENSUS_API_KEY is not None)

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 120)

# -----------------------------
# Paths / config
# -----------------------------
DATA_DIR = Path("data")
RAW_DIR = DATA_DIR / "raw"
OUT_DIR = DATA_DIR / "processed"
RAW_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Update this path if needed
IGS_PATH = r"C:\Users\jabba\Desktop\Code\machine_learning\AUC_mastercard_challenge\src\Inclusive_Growth_Score_Data_Export_25-02-2026_134202.csv"

# Tract selection for focused EDA / testing
TARGET_TRACTS = None
# Example:
# TARGET_TRACTS = ["13089023301", "13089023405", "13089023302"]

# If TARGET_TRACTS is set, the code will infer state FIPS from those tracts
SAVE_INTERMEDIATE = True

# ACS 5-year currently available through 2024
ACS_MAX_AVAILABLE_YEAR = 2024
FILL_2025_WITH_2024_ACS = False  # optional, keep False by default

Key loaded: True


### Cell 2 — IGS load / clean helpers

In [34]:
IGS_STRING_COLS = {
    "County",
    "State",
    "BENCHMARK",
    "Census Tract FIPS code",
    "geoid",
}

IGS_ALIAS_MAP = {
    "Inclusive Growth Score": "igs_total",
    "Growth": "igs_growth",
    "Inclusion": "igs_inclusion",
    "Place": "igs_place",
    "Economy": "igs_economy",
    "Community": "igs_community",
    "Place Growth": "igs_place_growth",
    "Place Inclusion": "igs_place_inclusion",
    "Economy Growth": "igs_economy_growth",
    "Economy Inclusion": "igs_economy_inclusion",
    "Community Growth": "igs_community_growth",
    "Community Inclusion": "igs_community_inclusion",
}


def dedupe_columns(cols):
    seen = {}
    out = []
    for c in cols:
        c = str(c).strip()
        if c not in seen:
            seen[c] = 0
            out.append(c)
        else:
            seen[c] += 1
            out.append(f"{c}.{seen[c]}")
    return out


def _find_header_row(df_raw, required_tokens=("Census Tract FIPS", "Year")):
    tokens = [t.lower() for t in required_tokens]
    for i in range(min(len(df_raw), 50)):
        row = df_raw.iloc[i].astype(str).str.lower().tolist()
        if all(any(tok in cell for cell in row) for tok in tokens):
            return i
    raise ValueError("Could not find the real IGS header row.")


def normalize_geoid_series(series: pd.Series) -> pd.Series:
    s = series.astype("string").str.strip()

    numeric = pd.to_numeric(s, errors="coerce")
    numeric_mask = numeric.notna()

    s = s.copy()
    s.loc[numeric_mask] = numeric.loc[numeric_mask].astype("Int64").astype("string")

    s = s.str.replace(r"\.0$", "", regex=True)
    s = s.str.replace(r"\D", "", regex=True)
    s = s.str.zfill(11)
    s = s.where(s.str.fullmatch(r"\d{11}"), pd.NA)

    return s


def filter_tracts(df: pd.DataFrame, tracts=None, geoid_col="geoid") -> pd.DataFrame:
    if not tracts:
        return df.copy()

    tract_set = {str(t).zfill(11) for t in tracts}
    out = df[df[geoid_col].astype("string").isin(tract_set)].copy()
    return out


def safe_save_parquet(df: pd.DataFrame, path: Path):
    out = df.copy()

    for c in out.columns:
        if out[c].dtype == "object":
            out[c] = out[c].astype("string")

    out.to_parquet(path, index=False, engine="pyarrow")


def load_igs_any(path: str) -> pd.DataFrame:
    p = Path(path)

    if p.suffix.lower() in [".xlsx", ".xls"]:
        raw = pd.read_excel(p, header=None)
    else:
        raw = pd.read_csv(p, header=None, low_memory=False)

    hdr_i = _find_header_row(raw, required_tokens=("Census Tract FIPS", "Year"))
    header = dedupe_columns(raw.iloc[hdr_i].tolist())

    df = raw.iloc[hdr_i + 1 :].copy()
    df.columns = header
    df = df.dropna(how="all").reset_index(drop=True)

    # Drop any duplicated header rows that still appear in the body
    if "Census Tract FIPS code" in df.columns:
        df = df[
            df["Census Tract FIPS code"].astype(str).str.strip().str.lower() != "census tract fips code"
        ].copy()

    # Normalize GEOID + year
    df["geoid"] = normalize_geoid_series(df["Census Tract FIPS code"])
    df["year"] = pd.to_numeric(df["Year"], errors="coerce")

    # Convert non-ID columns to numeric where possible
    for c in df.columns:
        if c not in IGS_STRING_COLS:
            df[c] = pd.to_numeric(df[c], errors="coerce")

    # Friendly aliases for easier modeling later
    for old_col, new_col in IGS_ALIAS_MAP.items():
        if old_col in df.columns:
            df[new_col] = pd.to_numeric(df[old_col], errors="coerce")

    if "Is an Opportunity Zone" in df.columns:
        df["is_opp_zone"] = pd.to_numeric(df["Is an Opportunity Zone"], errors="coerce")

    if "URBAN CODE" in df.columns:
        df["urban_code"] = pd.to_numeric(df["URBAN CODE"], errors="coerce")

    if "BENCHMARK" in df.columns:
        df["benchmark"] = df["BENCHMARK"].astype("string")

    # Final cleanup
    df = df.dropna(subset=["geoid", "year"]).copy()
    df["year"] = df["year"].astype(int)
    df = df.sort_values(["geoid", "year"]).reset_index(drop=True)

    return df

### Cell 3 — Load and save clean IGS

In [35]:
igs = load_igs_any(IGS_PATH)
igs = filter_tracts(igs, TARGET_TRACTS)

if SAVE_INTERMEDIATE:
    safe_save_parquet(igs, OUT_DIR / "igs_clean.parquet")

print("IGS shape:", igs.shape)
print("IGS years:", sorted(igs["year"].dropna().unique().tolist())[:10], "...", sorted(igs["year"].dropna().unique().tolist())[-3:])
print()
print(igs[["geoid", "year", "igs_total"]].head())
print()
print(igs[["igs_total", "igs_place", "igs_economy", "igs_community"]].describe())
print()
print("Top missingness:")
print(igs.isna().mean().sort_values(ascending=False).head(20))

IGS shape: (765288, 87)
IGS years: [2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025] ... [2023, 2024, 2025]

         geoid  year  igs_total
0  01001020100  2017       47.0
1  01001020100  2018       52.0
2  01001020100  2019       46.0
3  01001020100  2020       46.0
4  01001020100  2021       38.0

           igs_total      igs_place    igs_economy  igs_community
count  757582.000000  761912.000000  764294.000000  762407.000000
mean       50.052242      49.848921      50.328149      50.148915
std        10.252555      12.453479      13.532944      13.744805
min         0.000000       0.000000       0.000000       0.000000
25%        44.000000      42.000000      42.000000      40.000000
50%        50.000000      50.000000      51.500000      50.000000
75%        57.900000      58.000000      60.000000      60.000000
max        84.000000     100.000000      94.000000     100.000000

Top missingness:
Is an Opportunity Zone                      1.000000
Spend Growth Base, %        

### Cell 4 — ACS variables to pull

In [36]:
ACS_VAR_GROUPS = {
    "population_age": [
        "B01001_001E",  # total pop
        "B01001_003E", "B01001_004E", "B01001_005E", "B01001_006E",  # male under 18
        "B01001_027E", "B01001_028E", "B01001_029E", "B01001_030E",  # female under 18
        "B01001_020E", "B01001_021E", "B01001_022E", "B01001_023E", "B01001_024E", "B01001_025E",  # male 65+
        "B01001_044E", "B01001_045E", "B01001_046E", "B01001_047E", "B01001_048E", "B01001_049E",  # female 65+
    ],
    "race": [
        "B02001_001E",
        "B02001_002E",
        "B02001_003E",
        "B02001_005E",
        "B02001_008E",
    ],
    "education": [
        "B15003_001E",
        "B15003_022E", "B15003_023E", "B15003_024E", "B15003_025E",
    ],
    "employment": [
        "B23025_001E",
        "B23025_002E",
        "B23025_003E",
        "B23025_005E",
    ],
    "income_poverty": [
        "B19013_001E",  # median household income
        "B19301_001E",  # per-capita income
        "B19083_001E",  # gini
        "B17001_001E",  # poverty universe
        "B17001_002E",  # below poverty
    ],
    "commute": [
        "B08303_001E",
        "B08303_002E", "B08303_003E", "B08303_004E", "B08303_005E",
        "B08303_006E", "B08303_007E", "B08303_008E",
    ],
    "internet": [
        "B28002_001E",
        "B28002_002E",
    ],
    "housing": [
        "B25002_001E", "B25002_002E", "B25002_003E",   # total / occupied / vacant
        "B25003_001E", "B25003_002E", "B25003_003E",   # tenure
        "B25064_001E",                                  # median gross rent
        "B25077_001E",                                  # median home value
    ],
    "rent_burden": [
        "B25070_001E",
        "B25070_002E", "B25070_003E", "B25070_004E", "B25070_005E", "B25070_006E",
    ],
    "owner_cost_burden": [
        "B25091_001E",
        "B25091_003E", "B25091_004E", "B25091_005E", "B25091_006E", "B25091_007E",
        "B25091_014E", "B25091_015E", "B25091_016E", "B25091_017E", "B25091_018E",
    ],
    "early_education": [
        "B14003_004E", "B14003_013E", "B14003_032E", "B14003_041E",
    ],
    "health_insurance": [
        "B27001_001E",
        "B27001_005E", "B27001_008E", "B27001_011E", "B27001_014E", "B27001_017E",
        "B27001_020E", "B27001_023E", "B27001_026E", "B27001_029E",
        "B27001_033E", "B27001_036E", "B27001_039E", "B27001_042E", "B27001_045E",
        "B27001_048E", "B27001_051E", "B27001_054E", "B27001_057E",
    ],
}

ACS_VARS = sorted({v for group in ACS_VAR_GROUPS.values() for v in group})
print("ACS variable count:", len(ACS_VARS))

ACS variable count: 98


### Cell 5 — ACS API helpers

In [37]:
def chunks(lst, n):
    for i in range(0, len(lst), n):
        yield lst[i : i + n]


def infer_state_fips_from_tracts(tracts):
    if not tracts:
        return None
    return sorted({str(t).zfill(11)[:2] for t in tracts})


def resolve_acs_years(igs_years, max_available_year=ACS_MAX_AVAILABLE_YEAR):
    igs_years = sorted({int(y) for y in igs_years if pd.notna(y)})
    acs_years = [y for y in igs_years if y <= max_available_year]
    return acs_years


def get_state_fips(year: int, api_key: str) -> list[str]:
    url = f"https://api.census.gov/data/{year}/acs/acs5"
    params = {"get": "NAME", "for": "state:*", "key": api_key}

    r = requests.get(url, params=params, timeout=60)
    r.raise_for_status()

    data = r.json()
    header, rows = data[0], data[1:]
    df = pd.DataFrame(rows, columns=header)

    return sorted(df["state"].unique().tolist())


def census_get_tracts(year: int, state_fips: str, vars_: list[str], api_key: str) -> pd.DataFrame:
    base = f"https://api.census.gov/data/{year}/acs/acs5"
    get_str = "NAME," + ",".join(vars_)

    params = [
        ("get", get_str),
        ("for", "tract:*"),
        ("in", f"state:{state_fips}"),
        ("in", "county:*"),
        ("key", api_key),
    ]

    r = requests.get(base, params=params, timeout=120)
    r.raise_for_status()

    data = r.json()
    header, rows = data[0], data[1:]
    df = pd.DataFrame(rows, columns=header)
    df["year"] = year

    return df


def to_numeric_safe(df: pd.DataFrame, exclude=("NAME", "state", "county", "tract", "year")) -> pd.DataFrame:
    out = df.copy()
    for c in out.columns:
        if c in exclude:
            continue
        out[c] = pd.to_numeric(out[c], errors="coerce")
    return out

ACS_SENTINELS = {-666666666, -333333333, -222222222}

def clean_acs_values(df: pd.DataFrame, exclude=("NAME", "state", "county", "tract", "year")):
    out = df.copy()
    for c in out.columns:
        if c in exclude:
            continue
        out[c] = pd.to_numeric(out[c], errors="coerce")
        out[c] = out[c].replace(list(ACS_SENTINELS), np.nan)
    return out
    

def download_acs_all(
    years: list[int],
    api_key: str,
    state_fips_filter: list[str] | None = None,
    max_vars_per_call: int = 45,
    sleep_seconds: float = 0.25,
) -> pd.DataFrame:
    if not api_key:
        raise ValueError("Missing CENSUS_API_KEY. Put it in your environment before running ACS pulls.")

    all_parts = []

    for year in years:
        year_out = RAW_DIR / f"acs5_tract_{year}.parquet"

        if year_out.exists():
            print(f"[cache] {year_out.name}")
            df_year = pd.read_parquet(year_out)
            all_parts.append(df_year)
            continue

        states = state_fips_filter if state_fips_filter else get_state_fips(year, api_key)
        print(f"[download] year={year} | states={len(states)}")

        state_frames = []

        for st in states:
            merged_state = None

            for var_chunk in chunks(ACS_VARS, max_vars_per_call):
                df_chunk = census_get_tracts(year, st, var_chunk, api_key)
                df_chunk = clean_acs_values(df_chunk)

                keys = ["state", "county", "tract", "NAME", "year"]
                if merged_state is None:
                    merged_state = df_chunk
                else:
                    merged_state = merged_state.merge(df_chunk, on=keys, how="outer")

                time.sleep(sleep_seconds)

            state_frames.append(merged_state)

        df_year = pd.concat(state_frames, ignore_index=True)

        if SAVE_INTERMEDIATE:
            safe_save_parquet(df_year, year_out)

        all_parts.append(df_year)

    acs = pd.concat(all_parts, ignore_index=True)
    return acs

### Cell 6 — ACS feature engineering

In [38]:
def safe_div(n, d):
    n = np.asarray(n, dtype="float64")
    d = np.asarray(d, dtype="float64")

    out = np.full_like(n, np.nan, dtype="float64")
    valid = (~np.isnan(d)) & (d != 0)
    np.divide(n, d, out=out, where=valid)
    return out
    
def make_features(acs: pd.DataFrame) -> pd.DataFrame:
    acs = acs.copy()

    acs["state"] = acs["state"].astype(str).str.zfill(2)
    acs["county"] = acs["county"].astype(str).str.zfill(3)
    acs["tract"] = acs["tract"].astype(str).str.zfill(6)
    acs["geoid"] = acs["state"] + acs["county"] + acs["tract"]

    out = pd.DataFrame({
        "geoid": acs["geoid"],
        "year": acs["year"],
    })

    # -------------------------
    # Population / age
    # -------------------------
    out["pop_total"] = acs["B01001_001E"]

    out["pop_under18"] = (
        acs["B01001_003E"] + acs["B01001_004E"] + acs["B01001_005E"] + acs["B01001_006E"] +
        acs["B01001_027E"] + acs["B01001_028E"] + acs["B01001_029E"] + acs["B01001_030E"]
    )
    out["share_under18"] = safe_div(out["pop_under18"], out["pop_total"])

    out["pop_65plus"] = (
        acs["B01001_020E"] + acs["B01001_021E"] + acs["B01001_022E"] + acs["B01001_023E"] + acs["B01001_024E"] + acs["B01001_025E"] +
        acs["B01001_044E"] + acs["B01001_045E"] + acs["B01001_046E"] + acs["B01001_047E"] + acs["B01001_048E"] + acs["B01001_049E"]
    )
    out["share_65plus"] = safe_div(out["pop_65plus"], out["pop_total"])

    # -------------------------
    # Race shares
    # -------------------------
    out["share_white"] = safe_div(acs["B02001_002E"], acs["B02001_001E"])
    out["share_black"] = safe_div(acs["B02001_003E"], acs["B02001_001E"])
    out["share_asian"] = safe_div(acs["B02001_005E"], acs["B02001_001E"])
    out["share_two_plus"] = safe_div(acs["B02001_008E"], acs["B02001_001E"])

    # -------------------------
    # Education
    # -------------------------
    ba_plus = acs["B15003_022E"] + acs["B15003_023E"] + acs["B15003_024E"] + acs["B15003_025E"]
    out["ba_plus_share_25p"] = safe_div(ba_plus, acs["B15003_001E"])

    # -------------------------
    # Employment / economy
    # -------------------------
    out["lfpr_16p"] = safe_div(acs["B23025_002E"], acs["B23025_001E"])
    out["unemp_rate"] = safe_div(acs["B23025_005E"], acs["B23025_003E"])

    out["median_household_income"] = acs["B19013_001E"]
    out["per_capita_income"] = acs["B19301_001E"]
    out["gini"] = acs["B19083_001E"]
    out["poverty_rate"] = safe_div(acs["B17001_002E"], acs["B17001_001E"])

    # -------------------------
    # Commute / internet
    # -------------------------
    commute_under35 = (
        acs["B08303_002E"] + acs["B08303_003E"] + acs["B08303_004E"] + acs["B08303_005E"] +
        acs["B08303_006E"] + acs["B08303_007E"] + acs["B08303_008E"]
    )
    out["commute_under35_share"] = safe_div(commute_under35, acs["B08303_001E"])
    out["internet_sub_share"] = safe_div(acs["B28002_002E"], acs["B28002_001E"])

    # -------------------------
    # Housing / affordability
    # -------------------------
    out["housing_units_total"] = acs["B25002_001E"]
    out["occupied_units"] = acs["B25002_002E"]
    out["vacant_units"] = acs["B25002_003E"]
    out["occupied_share"] = safe_div(acs["B25002_002E"], acs["B25002_001E"])
    out["vacancy_rate"] = safe_div(acs["B25002_003E"], acs["B25002_001E"])

    out["owner_share"] = safe_div(acs["B25003_002E"], acs["B25003_001E"])
    out["renter_share"] = safe_div(acs["B25003_003E"], acs["B25003_001E"])

    out["median_gross_rent"] = acs["B25064_001E"]
    out["median_home_value"] = acs["B25077_001E"]

    rent_affordable = (
        acs["B25070_002E"] + acs["B25070_003E"] + acs["B25070_004E"] +
        acs["B25070_005E"] + acs["B25070_006E"]
    )

    owner_affordable = (
        acs["B25091_003E"] + acs["B25091_004E"] + acs["B25091_005E"] + acs["B25091_006E"] + acs["B25091_007E"] +
        acs["B25091_014E"] + acs["B25091_015E"] + acs["B25091_016E"] + acs["B25091_017E"] + acs["B25091_018E"]
    )

    denom_housing_cost = acs["B25070_001E"] + acs["B25091_001E"]
    out["affordable_housing_share"] = safe_div(rent_affordable + owner_affordable, denom_housing_cost)

    # -------------------------
    # Community / health
    # -------------------------
    # Early education proxy: enrolled 3-4 year olds over under-5 population
    enrolled_3_4 = acs["B14003_004E"] + acs["B14003_013E"] + acs["B14003_032E"] + acs["B14003_041E"]
    pop_under5 = acs["B01001_003E"] + acs["B01001_027E"]
    out["early_ed_enroll_share"] = safe_div(enrolled_3_4, pop_under5)

    uninsured = (
        acs["B27001_005E"] + acs["B27001_008E"] + acs["B27001_011E"] + acs["B27001_014E"] + acs["B27001_017E"] +
        acs["B27001_020E"] + acs["B27001_023E"] + acs["B27001_026E"] + acs["B27001_029E"] +
        acs["B27001_033E"] + acs["B27001_036E"] + acs["B27001_039E"] + acs["B27001_042E"] + acs["B27001_045E"] +
        acs["B27001_048E"] + acs["B27001_051E"] + acs["B27001_054E"] + acs["B27001_057E"]
    )
    out["insured_share"] = 1.0 - safe_div(uninsured, acs["B27001_001E"])

    # -------------------------
    # Growth features
    # -------------------------
    out = out.sort_values(["geoid", "year"]).reset_index(drop=True)

    growth_cols = [
        "median_household_income",
        "per_capita_income",
        "occupied_units",
        "median_home_value",
        "median_gross_rent",
    ]

    for col in growth_cols:
        out[f"{col}_growth"] = out.groupby("geoid")[col].pct_change(fill_method=None)

    return out

### Cell 7 — Merge helpers

In [39]:
def prepare_acs_raw_for_merge(acs: pd.DataFrame) -> pd.DataFrame:
    acs_raw = acs.copy()

    acs_raw["state"] = acs_raw["state"].astype(str).str.zfill(2)
    acs_raw["county"] = acs_raw["county"].astype(str).str.zfill(3)
    acs_raw["tract"] = acs_raw["tract"].astype(str).str.zfill(6)
    acs_raw["geoid"] = acs_raw["state"] + acs_raw["county"] + acs_raw["tract"]

    # one row per tract-year expected
    acs_raw = acs_raw.drop_duplicates(subset=["geoid", "year"]).copy()

    return acs_raw


def merge_igs_acs_raw(igs: pd.DataFrame, acs_raw: pd.DataFrame) -> pd.DataFrame:
    merged = igs.merge(acs_raw, on=["geoid", "year"], how="left", validate="m:1")
    return merged


def merge_igs_acs_full(
    igs: pd.DataFrame,
    acs_raw: pd.DataFrame,
    feats: pd.DataFrame,
    
) -> pd.DataFrame:
    merged = (
        igs
        .merge(acs_raw, on=["geoid", "year"], how="left", validate="m:1")
        .merge(feats, on=["geoid", "year"], how="left", validate="m:1")
    )
    return merged

In [40]:
# for year in range(2017, 2025):
#     p = RAW_DIR / f"acs5_tract_{year}.parquet"
#     if p.exists():
#         p.unlink()
#         print("deleted", p.name)

### Cell 8 — Run ACS pull, feature engineering, and merge

In [41]:
igs_years = sorted(igs["year"].dropna().unique().tolist())
acs_years = resolve_acs_years(igs_years, max_available_year=ACS_MAX_AVAILABLE_YEAR)
state_fips_filter = infer_state_fips_from_tracts(TARGET_TRACTS)

print("IGS years in file:", igs_years)
print("ACS years to pull:", acs_years)
print("State filter from target tracts:", state_fips_filter)

acs = download_acs_all(
    years=acs_years,
    api_key=CENSUS_API_KEY,
    state_fips_filter=state_fips_filter,
    max_vars_per_call=45,
    sleep_seconds=0.25,
)

acs_raw = prepare_acs_raw_for_merge(acs)
feats = make_features(acs)
feats = filter_tracts(feats, TARGET_TRACTS)

# keep only IGS years that exist in ACS-derived tables
valid_years = sorted(feats["year"].dropna().unique())
model_igs = igs[igs["year"].isin(valid_years)].copy()

igs_x_acs_raw = merge_igs_acs_raw(model_igs, acs_raw)
model_df_full = merge_igs_acs_full(model_igs, acs_raw, feats)

if SAVE_INTERMEDIATE:
    safe_save_parquet(acs, OUT_DIR / "acs_raw.parquet")
    safe_save_parquet(acs_raw, OUT_DIR / "acs_raw_with_geoid.parquet")
    safe_save_parquet(feats, OUT_DIR / "acs_features.parquet")
    safe_save_parquet(igs_x_acs_raw, OUT_DIR / "igs_x_acs_raw.parquet")
    safe_save_parquet(model_df_full, OUT_DIR / "igs_x_acs_full.parquet")

print("ACS raw shape:", acs.shape)
print("ACS raw+geoid shape:", acs_raw.shape)
print("ACS features shape:", feats.shape)
print("IGS x ACS raw shape:", igs_x_acs_raw.shape)
print("Full merged model_df_full shape:", model_df_full.shape)

IGS years in file: [2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]
ACS years to pull: [2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024]
State filter from target tracts: None
[cache] acs5_tract_2017.parquet
[cache] acs5_tract_2018.parquet
[cache] acs5_tract_2019.parquet
[cache] acs5_tract_2020.parquet
[cache] acs5_tract_2021.parquet
[cache] acs5_tract_2022.parquet
[cache] acs5_tract_2023.parquet
[cache] acs5_tract_2024.parquet
ACS raw shape: (648952, 103)
ACS raw+geoid shape: (648952, 104)
ACS features shape: (648952, 37)
IGS x ACS raw shape: (680256, 189)
Full merged model_df_full shape: (680256, 224)


### validation cell right after Cell 8

In [42]:
# choose the final EDA panel
eda_df = model_df_full.copy()

print("EDA shape:", eda_df.shape)
print("EDA years:", sorted(eda_df["year"].dropna().unique().tolist()))
print("EDA unique tracts:", eda_df["geoid"].nunique())

# required ACS raw columns
required_acs_raw = ["NAME", "state", "county", "tract"] + ACS_VARS
missing_acs_raw = [c for c in required_acs_raw if c not in eda_df.columns]

# required engineered columns
required_feats = [c for c in feats.columns if c not in ["geoid", "year"]]
missing_feats = [c for c in required_feats if c not in eda_df.columns]

# all original IGS columns
missing_igs = [c for c in igs.columns if c not in eda_df.columns]

print("Missing raw ACS columns:", missing_acs_raw[:20], "count =", len(missing_acs_raw))
print("Missing engineered feature columns:", missing_feats[:20], "count =", len(missing_feats))
print("Missing IGS columns:", missing_igs[:20], "count =", len(missing_igs))

print("Duplicate geoid-year rows:", eda_df.duplicated(subset=["geoid", "year"]).sum())

# quick sample
cols_to_show = [
    "geoid", "year", "igs_total", "igs_place", "igs_economy", "igs_community",
    "NAME", "state", "county", "tract",
    "B19013_001E", "B19301_001E", "B25077_001E",
    "median_household_income", "per_capita_income", "median_home_value"
]
print(eda_df[cols_to_show].head())

EDA shape: (680256, 224)
EDA years: [2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024]
EDA unique tracts: 85032
Missing raw ACS columns: [] count = 0
Missing engineered feature columns: [] count = 0
Missing IGS columns: [] count = 0
Duplicate geoid-year rows: 0
         geoid  year  igs_total  igs_place  igs_economy  igs_community                                       NAME state county   tract  B19013_001E  B19301_001E  B25077_001E  \
0  01001020100  2017       47.0       53.0         34.0           54.0  Census Tract 201, Autauga County, Alabama    01    001  020100      67826.0      33018.0     152500.0   
1  01001020100  2018       52.0       48.0         48.0           62.0  Census Tract 201, Autauga County, Alabama    01    001  020100      58625.0      31580.0     133300.0   
2  01001020100  2019       46.0       40.0         40.0           56.0  Census Tract 201, Autauga County, Alabama    01    001  020100      60208.0      31225.0     136100.0   
3  01001020100  2020       46.0 

### Cell 9 — Final checks before EDA

In [43]:
print("Merged years:", sorted(model_df_full["year"].dropna().unique().tolist()))
print()

print(model_df_full[[
    "geoid", "year", "igs_total", "igs_place", "igs_economy", "igs_community",
    "affordable_housing_share", "internet_sub_share", "insured_share",
    "median_household_income", "poverty_rate", "vacancy_rate"
]].head())

print()
print("Top missingness in merged data:")
print(model_df_full.isna().mean().sort_values(ascending=False).head(30))

print()
print("Core numeric summary:")
core_cols = [
    "igs_total",
    "igs_place",
    "igs_economy",
    "igs_community",
    "affordable_housing_share",
    "internet_sub_share",
    "insured_share",
    "median_household_income",
    "per_capita_income",
    "poverty_rate",
    "vacancy_rate",
    "unemp_rate",
]
print(model_df_full[core_cols].describe())

Merged years: [2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024]

         geoid  year  igs_total  igs_place  igs_economy  igs_community  affordable_housing_share  internet_sub_share  insured_share  median_household_income  poverty_rate  vacancy_rate
0  01001020100  2017       47.0       53.0         34.0           54.0                  0.737401            0.762599       0.909485                  67826.0      0.106775      0.014379
1  01001020100  2018       52.0       48.0         48.0           62.0                  0.704575            0.717647       0.907436                  58625.0      0.113365      0.017972
2  01001020100  2019       46.0       40.0         40.0           56.0                  0.715092            0.740480       0.902659                  60208.0      0.166583      0.022069
3  01001020100  2020       46.0       34.0         48.0           56.0                  0.738817            0.810967       0.903658                  60388.0      0.136528      0.023944
4  01001020

###  Cell 10 — Optional: save an EDA subset for only chosen tracts

In [44]:
# Cell 10 — final EDA panel cleanup + save

eda_df = model_df_full.copy()   # use model_df_full, not model_df

# drop junk unnamed / NaN column names from the original IGS export
junk_cols = [c for c in eda_df.columns if pd.isna(c) or str(c).startswith("Unnamed")]
print("Dropping junk columns:", junk_cols)

eda_df = eda_df.drop(columns=junk_cols, errors="ignore")

# optional: reorder key columns to the front
front_cols = [
    c for c in [
        "geoid", "year", "Census Tract FIPS code", "County", "State",
        "igs_total", "igs_place", "igs_economy", "igs_community"
    ] if c in eda_df.columns
]
other_cols = [c for c in eda_df.columns if c not in front_cols]
eda_df = eda_df[front_cols + other_cols]

print("EDA panel shape:", eda_df.shape)

if SAVE_INTERMEDIATE:
    safe_save_parquet(eda_df, OUT_DIR / "eda_panel_clean.parquet")

eda_df.head()

Dropping junk columns: []
EDA panel shape: (680256, 224)


,geoid,year,Census Tract FIPS code,County,State,igs_total,igs_place,igs_economy,igs_community,nan,Is an Opportunity Zone,Year,Inclusive Growth Score,Growth,Inclusion,Place,Place Growth,Place Inclusion,Net Occupancy Score,"Net Occupancy Base, %","Net Occupancy Tract, %",Residential Real Estate Value Score,"Residential Real Estate Value Base, %","Residential Real Estate Value Tract, %",Acres of Park Land Score,"Acres of Park Land Base, %","Acres of Park Land Tract, %",Affordable Housing Score,"Affordable Housing Base, %","Affordable Housing Tract, %",Internet Access Score,"Internet Access Base, %","Internet Access Tract, %",Travel Time to Work Score,"Travel Time to Work Base, %","Travel Time to Work Tract, %",Economy,Economy Growth,Economy Inclusion,New Businesses Score,"New Businesses Base, %","New Businesses Tract, %",Spend Growth Score,"Spend Growth Base, %","Spend Growth Tract, %",Small Business Loans Score,"Small Business Loans Base, %","Small Business Loans Tract, %",Minority/Women Owned Businesses Score,"Minority/Women Owned Businesses Base, %","Minority/Women Owned Businesses Tract, %",Labor Market Engagement Index Score,Labor Market Engagement Index Base,Labor Market Engagement Index Tract,Commercial Diversity Score,"Commercial Diversity Base, %","Commercial Diversity Tract, %",Community,Community Growth,Community Inclusion,Personal Income Score,"Personal Income Base, %","Personal Income Tract, %",Spending per Capita Score,"Spending per Capita Base, %","Spending per Capita Tract, %",Female Above Poverty Score,"Female Above Poverty Base, %","Female Above Poverty Tract, %",Gini Coefficient Score,Gini Coefficient Base,Gini Coefficient Tract,Early Education Enrollment Score,"Early Education Enrollment Base, %","Early Education Enrollment Tract, %",Health Insurance Coverage Score,"Health Insurance Coverage Base, %","Health Insurance Coverage Tract, %",igs_growth,igs_inclusion,igs_place_growth,igs_place_inclusion,igs_economy_growth,igs_economy_inclusion,igs_community_growth,igs_community_inclusion,is_opp_zone,NAME,B01001_001E,B01001_003E,B01001_004E,B01001_005E,B01001_006E,B01001_020E,B01001_021E,B01001_022E,B01001_023E,B01001_024E,B01001_025E,B01001_027E,...,B14003_032E,B14003_041E,B15003_001E,B15003_022E,B15003_023E,B15003_024E,B15003_025E,B17001_001E,B17001_002E,state,county,tract,B19013_001E,B19083_001E,B19301_001E,B23025_001E,B23025_002E,B23025_003E,B23025_005E,B25002_001E,B25002_002E,B25002_003E,B25003_001E,B25003_002E,B25003_003E,B25064_001E,B25070_001E,B25070_002E,B25070_003E,B25070_004E,B25070_005E,B25070_006E,B25077_001E,B25091_001E,B25091_003E,B25091_004E,B25091_005E,B25091_006E,B25091_007E,B25091_014E,B25091_015E,B25091_016E,B25091_017E,B25091_018E,B27001_001E,B27001_005E,B27001_008E,B27001_011E,B27001_014E,B27001_017E,B27001_020E,B27001_023E,B27001_026E,B27001_029E,B27001_033E,B27001_036E,B27001_039E,B27001_042E,B27001_045E,B27001_048E,B27001_051E,B27001_054E,B27001_057E,B28002_001E,B28002_002E,pop_total,pop_under18,share_under18,pop_65plus,share_65plus,share_white,share_black,share_asian,share_two_plus,ba_plus_share_25p,lfpr_16p,unemp_rate,median_household_income,per_capita_income,gini,poverty_rate,commute_under35_share,internet_sub_share,housing_units_total,occupied_units,vacant_units,occupied_share,vacancy_rate,owner_share,renter_share,median_gross_rent,median_home_value,affordable_housing_share,early_ed_enroll_share,insured_share,median_household_income_growth,per_capita_income_growth,occupied_units_growth,median_home_value_growth,median_gross_rent_growth
0,01001020100,2017,01001020100,Autauga County,Alabama,47.0,53.0,34.0,54.0,8100.0,NaN,2017,47.0,45.0,49.0,53.0,62.0,44.0,62.0,1.5,11.4,63.0,-1.2,15.7,29.0,3.1,1.4,98.0,76.5,90.2,43.0,78.6,76.3,5.0,74.1,46.1,34.0,30.0,38.0,67.0,4.1,57.1,10.0,NaN,NaN,14.0,4.5,-15.8,0.0,0.0,0.0,67.0,48.0,65.0,8.0,20.3,10.5,54.0,42.0,66.0,33.0,3.0,-4.2,52.0,NaN,NaN,78.0,83.7,91.6,44.0,41.4,42.3,96.0,23.1,53.2,44.0,90.0,88.9,45.0,49.0,62.0,44.0,30.0,38.0,42.0,66.0,NaN,"Census Trac

### Cell 11 — validation checks


In [45]:
# Cell 11 — validation checks

print("Duplicate geoid-year rows:", eda_df.duplicated(subset=["geoid", "year"]).sum())

required_acs_raw = ["NAME", "state", "county", "tract"] + ACS_VARS
missing_acs_raw = [c for c in required_acs_raw if c not in eda_df.columns]

required_feats = [c for c in feats.columns if c not in ["geoid", "year"]]
missing_feats = [c for c in required_feats if c not in eda_df.columns]

missing_igs = [c for c in igs.columns if c not in eda_df.columns]

print("Missing raw ACS columns:", len(missing_acs_raw))
print("Missing engineered ACS columns:", len(missing_feats))
print("Missing IGS columns:", len(missing_igs))

Duplicate geoid-year rows: 0
Missing raw ACS columns: 0
Missing engineered ACS columns: 0
Missing IGS columns: 0


### Cell 12 — model settings

In [46]:
# Cell 12 — trajectory-first model settings

import os
import zipfile
import requests
from pathlib import Path
from itertools import combinations
from collections import defaultdict, deque

import numpy as np
import pandas as pd
import geopandas as gpd

# Analysis windows
PRE_YEARS = [2017, 2018, 2019]
SHOCK_YEARS = [2020, 2021]
RECOVERY_YEARS = [2022, 2023, 2024]
ANALYSIS_YEARS = PRE_YEARS + SHOCK_YEARS + RECOVERY_YEARS
CURRENT_YEAR = 2024

# Challenge-aligned thresholds
LOW_IGS_THRESHOLD = 45

# Economic vulnerability rules (tract-level and cluster-level)
ECON_VULN_PCTL = 0.40
ECON_VULN_MIN_CONDS = 3

# Adjacency / cluster rules
ADJACENCY_MODE = "shared_edge_or_corner"
MIN_CLUSTER_TRACTS = 3
MIN_CLUSTER_POP = 5000

# Tract / quality flags
TINY_TRACT_POP = 1500
MAX_TINY_TRACT_SHARE = 0.75
MAX_ZERO_IGS_SHARE = 0.50

# Geometry paths
TRACT_BASE_URL = "https://www2.census.gov/geo/tiger/TIGER2024/TRACT/"
TRACT_ROOT_DIR = RAW_DIR / "tl_2024_state_tracts"
TRACT_ROOT_DIR.mkdir(parents=True, exist_ok=True)

print("Analysis years:", ANALYSIS_YEARS)
print("Pre-COVID years:", PRE_YEARS)
print("Shock years:", SHOCK_YEARS)
print("Recovery years:", RECOVERY_YEARS)
print("Low IGS threshold:", LOW_IGS_THRESHOLD)
print("Economic vulnerability minimum conditions:", ECON_VULN_MIN_CONDS)
print("Adjacency mode:", ADJACENCY_MODE)

Analysis years: [2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024]
Pre-COVID years: [2017, 2018, 2019]
Shock years: [2020, 2021]
Recovery years: [2022, 2023, 2024]
Low IGS threshold: 45
Economic vulnerability minimum conditions: 3
Adjacency mode: shared_edge_or_corner


### Cell 13 — helper functions

In [47]:
# Cell 13 — helper functions

def weighted_mean(values: pd.Series, weights: pd.Series) -> float:
    vals = pd.to_numeric(values, errors="coerce")
    wts = pd.to_numeric(weights, errors="coerce")
    mask = vals.notna() & wts.notna() & (wts > 0)
    if mask.sum() == 0:
        return np.nan
    return np.average(vals.loc[mask], weights=wts.loc[mask])

def percentile_score(series: pd.Series, weight: float, higher_is_better: bool) -> pd.Series:
    """
    Convert a metric into weighted percentile points.
    - higher_is_better=True  -> larger values get more points
    - higher_is_better=False -> smaller values get more points
    """
    ranked = series.rank(
        method="average",
        pct=True,
        ascending=higher_is_better
    )
    return ranked * weight

def add_period_means(df: pd.DataFrame, value_cols: list[str], years: list[int], prefix: str) -> pd.DataFrame:
    out = (
        df.loc[df["year"].isin(years), ["geoid", "year"] + value_cols]
        .groupby("geoid", as_index=False)[value_cols]
        .mean()
        .rename(columns={c: f"{prefix}_{c}" for c in value_cols})
    )
    return out

def add_cluster_period_means(df: pd.DataFrame, value_cols: list[str], years: list[int], prefix: str) -> pd.DataFrame:
    out = (
        df.loc[df["year"].isin(years), ["cluster_id", "year"] + value_cols]
        .groupby("cluster_id", as_index=False)[value_cols]
        .mean()
        .rename(columns={c: f"{prefix}_{c}" for c in value_cols})
    )
    return out

def connected_components_from_edges(nodes, edges_df):
    neighbors = defaultdict(set)
    for n in nodes:
        neighbors[n] = set()

    for _, row in edges_df.iterrows():
        a = row["geoid_1"]
        b = row["geoid_2"]
        neighbors[a].add(b)
        neighbors[b].add(a)

    visited = set()
    node_to_cluster = {}
    cluster_id = 0

    for n in nodes:
        if n in visited:
            continue

        cluster_id += 1
        q = deque([n])
        visited.add(n)

        while q:
            cur = q.popleft()
            node_to_cluster[cur] = cluster_id
            for nxt in neighbors[cur]:
                if nxt not in visited:
                    visited.add(nxt)
                    q.append(nxt)

    return node_to_cluster

def build_touch_edges(gdf: gpd.GeoDataFrame) -> pd.DataFrame:
    """
    Shared edge/corner adjacency using geometry.touches().
    Runs county-by-county for tract candidates.
    """
    edge_parts = []

    for (state_name, county_name), grp in gdf.groupby(["display_state", "display_county"], dropna=False):
        grp = grp[["geoid", "geometry"]].drop_duplicates("geoid").copy()
        if len(grp) < 2:
            continue

        left = grp.rename(columns={"geoid": "geoid_left"})
        right = grp.rename(columns={"geoid": "geoid_right"})

        joined = gpd.sjoin(left, right, how="inner", predicate="touches")
        joined = joined[joined["geoid_left"] < joined["geoid_right"]].copy()

        if len(joined) == 0:
            continue

        joined["display_state"] = state_name
        joined["display_county"] = county_name

        edge_parts.append(
            joined[["display_state", "display_county", "geoid_left", "geoid_right"]]
            .rename(columns={"geoid_left": "geoid_1", "geoid_right": "geoid_2"})
        )

    if edge_parts:
        edges = pd.concat(edge_parts, ignore_index=True).drop_duplicates().reset_index(drop=True)
    else:
        edges = pd.DataFrame(columns=["display_state", "display_county", "geoid_1", "geoid_2"])

    return edges

### Cell 14 — working panel and required columns

In [48]:
# Cell 14 — working panel and required columns

required_model_cols = [
    "geoid", "year",
    "igs_total", "igs_economy",
    "poverty_rate", "unemp_rate", "median_household_income", "lfpr_16p",
    "pop_total",
    "NAME", "state", "county", "tract", "State", "County"
]

missing_model_cols = [c for c in required_model_cols if c not in eda_df.columns]
print("Missing model columns:", missing_model_cols)
if missing_model_cols:
    raise ValueError(f"Missing required model columns: {missing_model_cols}")

model_df = eda_df.loc[eda_df["year"].isin(ANALYSIS_YEARS)].copy()

model_df["display_state"] = model_df["State"].fillna(model_df["state"])
model_df["display_county"] = model_df["County"].fillna(model_df["county"])
model_df["display_name"] = model_df["NAME"]
model_df["tiny_tract_flag"] = model_df["pop_total"] < TINY_TRACT_POP
model_df["zero_igs_flag"] = model_df["igs_total"].fillna(np.nan).eq(0)

print("Model panel shape:", model_df.shape)
print("Model years:", sorted(model_df["year"].dropna().unique().tolist()))
print("Unique tracts:", model_df["geoid"].nunique())

Missing model columns: []
Model panel shape: (680256, 229)
Model years: [2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024]
Unique tracts: 85032


### Cell 15 — tract-level time summary

In [49]:
# Cell 15 — tract-level time summary

tract_id_cols = ["geoid", "display_state", "display_county", "display_name"]
tract_time_cols = [
    "igs_total",
    "igs_economy",
    "poverty_rate",
    "unemp_rate",
    "median_household_income",
    "lfpr_16p",
    "pop_total",
]

tract_base = model_df[tract_id_cols].drop_duplicates(subset=["geoid"]).copy()

tract_pre = add_period_means(model_df, tract_time_cols, PRE_YEARS, "pre")
tract_shock = add_period_means(model_df, tract_time_cols, SHOCK_YEARS, "shock")
tract_recovery = add_period_means(model_df, tract_time_cols, RECOVERY_YEARS, "recovery")

tract_2024 = (
    model_df.loc[model_df["year"] == CURRENT_YEAR, ["geoid"] + tract_time_cols]
    .groupby("geoid", as_index=False)[tract_time_cols]
    .mean()
    .rename(columns={c: f"y2024_{c}" for c in tract_time_cols})
)

tract_year_counts = (
    model_df.groupby("geoid", as_index=False)
    .agg(
        years_observed=("year", "nunique"),
        years_below_45=("igs_total", lambda s: int((s < LOW_IGS_THRESHOLD).sum())),
    )
)

tract_time_summary = (
    tract_base
    .merge(tract_pre, on="geoid", how="left")
    .merge(tract_shock, on="geoid", how="left")
    .merge(tract_recovery, on="geoid", how="left")
    .merge(tract_2024, on="geoid", how="left")
    .merge(tract_year_counts, on="geoid", how="left")
    .copy()
)

# trajectory metrics
tract_time_summary["igs_total_shock_drop"] = tract_time_summary["shock_igs_total"] - tract_time_summary["pre_igs_total"]
tract_time_summary["igs_total_recovery_gap"] = tract_time_summary["recovery_igs_total"] - tract_time_summary["pre_igs_total"]
tract_time_summary["igs_economy_shock_drop"] = tract_time_summary["shock_igs_economy"] - tract_time_summary["pre_igs_economy"]
tract_time_summary["igs_economy_recovery_gap"] = tract_time_summary["recovery_igs_economy"] - tract_time_summary["pre_igs_economy"]

tract_time_summary["recent_low_igs_flag"] = (
    (tract_time_summary["recovery_igs_total"] < LOW_IGS_THRESHOLD) |
    (tract_time_summary["y2024_igs_total"] < LOW_IGS_THRESHOLD)
)

print("Tract time summary shape:", tract_time_summary.shape)
print(tract_time_summary.head())

Tract time summary shape: (85032, 39)
         geoid display_state  display_county                               display_name  pre_igs_total  pre_igs_economy  pre_poverty_rate  pre_unemp_rate  pre_median_household_income  pre_lfpr_16p  \
0  01001020100       Alabama  Autauga County  Census Tract 201, Autauga County, Alabama      48.333333        40.666667          0.128908        0.036769                 62219.666667      0.615629   
1  01001020200       Alabama  Autauga County  Census Tract 202, Autauga County, Alabama      42.000000        50.000000          0.198082        0.040750                 42925.333333      0.496928   
2  01001020300       Alabama  Autauga County  Census Tract 203, Autauga County, Alabama      45.666667        52.000000          0.156500        0.038299                 51342.000000      0.616845   
3  01001020400       Alabama  Autauga County  Census Tract 204, Autauga County, Alabama      50.333333        54.000000          0.032612        0.040518         

### Cell 15.5 — shortlist audit

In [50]:
# # Cell 15.5 — shortlist audit

# print("=== SHORTLIST AUDIT ===")
# print()

# # 1) Row-count audit
# eligible_n = len(shortlist_base)
# scored_n = len(tract_shortlist_scored)
# dropped_n = eligible_n - scored_n
# dropped_pct = (dropped_n / eligible_n * 100) if eligible_n else 0

# print("Row-count audit")
# print(f"Eligible low-IGS tracts in shortlist_base: {eligible_n:,}")
# print(f"Scored tracts in tract_shortlist_scored: {scored_n:,}")
# print(f"Dropped before scoring: {dropped_n:,} ({dropped_pct:.2f}%)")
# print()

# # 2) Missingness audit for the scoring inputs
# print("Missingness audit on score_input_cols within shortlist_base")
# missing_summary = pd.DataFrame({
#     "missing_count": shortlist_base[score_input_cols].isna().sum(),
#     "missing_pct": shortlist_base[score_input_cols].isna().mean().mul(100)
# }).sort_values(["missing_count", "missing_pct"], ascending=[False, False])

# print(missing_summary)
# print()

# # 3) Which columns are causing drops specifically?
# dropped_rows = shortlist_base[shortlist_base[score_input_cols].isna().any(axis=1)].copy()

# print("Rows dropped because at least one score input is missing:", len(dropped_rows))
# print()

# if len(dropped_rows) > 0:
#     dropped_missing_summary = pd.DataFrame({
#         "missing_count_among_dropped": dropped_rows[score_input_cols].isna().sum(),
#         "missing_pct_among_dropped": dropped_rows[score_input_cols].isna().mean().mul(100)
#     }).sort_values(["missing_count_among_dropped", "missing_pct_among_dropped"], ascending=[False, False])

#     print("Missingness among dropped rows only")
#     print(dropped_missing_summary)
#     print()

#     print("Sample dropped rows")
#     display_cols = [
#         "geoid", "display_state", "display_county", "display_name",
#         "igs_total", "igs_place", "igs_economy", "igs_community"
#     ] + score_input_cols

#     print(dropped_rows[display_cols].head(10))
#     print()

# # 4) Score-construction audit
# print("Score construction audit")

# tract_shortlist_scored["igs_distress_score_check"] = (
#     tract_shortlist_scored["igs_total_pts"] +
#     tract_shortlist_scored["igs_economy_pts"] +
#     tract_shortlist_scored["igs_place_pts"] +
#     tract_shortlist_scored["igs_community_pts"] +
#     tract_shortlist_scored["persistence_score"]
# )

# tract_shortlist_scored["healthcare_proxy_score_check"] = (
#     tract_shortlist_scored["poverty_rate_pts"] +
#     tract_shortlist_scored["insured_share_pts"] +
#     tract_shortlist_scored["median_household_income_pts"] +
#     tract_shortlist_scored["unemp_rate_pts"] +
#     tract_shortlist_scored["affordable_housing_share_pts"] +
#     tract_shortlist_scored["internet_sub_share_pts"]
# )

# tract_shortlist_scored["vulnerability_score_check"] = (
#     tract_shortlist_scored["share_65plus_pts"] +
#     tract_shortlist_scored["share_under18_pts"] +
#     tract_shortlist_scored["lfpr_16p_pts"] +
#     tract_shortlist_scored["vacancy_rate_pts"]
# )

# tract_shortlist_scored["total_shortlist_score_check"] = (
#     tract_shortlist_scored["igs_distress_score_check"] +
#     tract_shortlist_scored["healthcare_proxy_score_check"] +
#     tract_shortlist_scored["vulnerability_score_check"]
# )

# score_audit = pd.DataFrame({
#     "igs_distress_max_abs_diff": [
#         (tract_shortlist_scored["igs_distress_score"] - tract_shortlist_scored["igs_distress_score_check"]).abs().max()
#     ],
#     "healthcare_proxy_max_abs_diff": [
#         (tract_shortlist_scored["healthcare_proxy_score"] - tract_shortlist_scored["healthcare_proxy_score_check"]).abs().max()
#     ],
#     "vulnerability_max_abs_diff": [
#         (tract_shortlist_scored["vulnerability_score"] - tract_shortlist_scored["vulnerability_score_check"]).abs().max()
#     ],
#     "total_score_max_abs_diff": [
#         (tract_shortlist_scored["total_shortlist_score"] - tract_shortlist_scored["total_shortlist_score_check"]).abs().max()
#     ],
# })

# print(score_audit)
# print()

# # 5) Score-range audit
# print("Score range audit")
# range_audit = pd.DataFrame({
#     "min": tract_shortlist_scored[
#         ["igs_distress_score", "healthcare_proxy_score", "vulnerability_score", "total_shortlist_score"]
#     ].min(),
#     "max": tract_shortlist_scored[
#         ["igs_distress_score", "healthcare_proxy_score", "vulnerability_score", "total_shortlist_score"]
#     ].max(),
#     "mean": tract_shortlist_scored[
#         ["igs_distress_score", "healthcare_proxy_score", "vulnerability_score", "total_shortlist_score"]
#     ].mean(),
# })
# print(range_audit)
# print()

# # 6) Persistence audit
# print("Persistence audit")
# print(tract_shortlist_scored["n_persist_years"].value_counts(dropna=False).sort_index())
# print()
# print("Persistence score min/max:",
#       tract_shortlist_scored["persistence_score"].min(),
#       tract_shortlist_scored["persistence_score"].max())
# print()

# # 7) Tiny-tract audit among top-ranked rows
# print("Tiny-tract audit in top 25 scored rows")
# print(
#     tract_shortlist_scored.head(25)[
#         ["geoid", "display_state", "display_county", "igs_total", "total_shortlist_score", "tiny_tract_flag"]
#     ]
# )
# print()

# print("=== END SHORTLIST AUDIT ===")

### Cell 16 — tract-level economic vulnerability screen

In [51]:
# Cell 16 — tract-level economic vulnerability screen

# percentiles are based on recovery-period tract values
q_igs_econ_low = tract_time_summary["recovery_igs_economy"].quantile(ECON_VULN_PCTL)
q_poverty_high = tract_time_summary["recovery_poverty_rate"].quantile(1 - ECON_VULN_PCTL)
q_unemp_high = tract_time_summary["recovery_unemp_rate"].quantile(1 - ECON_VULN_PCTL)
q_income_low = tract_time_summary["recovery_median_household_income"].quantile(ECON_VULN_PCTL)
q_lfpr_low = tract_time_summary["recovery_lfpr_16p"].quantile(ECON_VULN_PCTL)

tract_time_summary["econ_cond_low_igs_economy"] = tract_time_summary["recovery_igs_economy"] <= q_igs_econ_low
tract_time_summary["econ_cond_high_poverty"] = tract_time_summary["recovery_poverty_rate"] >= q_poverty_high
tract_time_summary["econ_cond_high_unemp"] = tract_time_summary["recovery_unemp_rate"] >= q_unemp_high
tract_time_summary["econ_cond_low_income"] = tract_time_summary["recovery_median_household_income"] <= q_income_low
tract_time_summary["econ_cond_low_lfpr"] = tract_time_summary["recovery_lfpr_16p"] <= q_lfpr_low

econ_cond_cols = [
    "econ_cond_low_igs_economy",
    "econ_cond_high_poverty",
    "econ_cond_high_unemp",
    "econ_cond_low_income",
    "econ_cond_low_lfpr",
]

tract_time_summary["econ_vuln_cond_count"] = tract_time_summary[econ_cond_cols].sum(axis=1)
tract_time_summary["econ_vulnerable_flag"] = tract_time_summary["econ_vuln_cond_count"] >= ECON_VULN_MIN_CONDS

# broad initial tract pool:
# low IGS now/recently + economically vulnerable
tract_candidate_summary = tract_time_summary.loc[
    tract_time_summary["recent_low_igs_flag"] & tract_time_summary["econ_vulnerable_flag"]
].copy()

tract_candidate_pool_2024 = (
    model_df.loc[model_df["year"] == CURRENT_YEAR]
    .merge(
        tract_candidate_summary[
            [
                "geoid",
                "pre_igs_total", "shock_igs_total", "recovery_igs_total", "y2024_igs_total",
                "pre_igs_economy", "shock_igs_economy", "recovery_igs_economy", "y2024_igs_economy",
                "econ_vuln_cond_count", "econ_vulnerable_flag",
                "igs_total_shock_drop", "igs_total_recovery_gap",
                "igs_economy_shock_drop", "igs_economy_recovery_gap",
                "years_observed", "years_below_45"
            ]
        ],
        on="geoid",
        how="inner",
        validate="1:1"
    )
    .copy()
)

print("Candidate tract summary shape:", tract_candidate_summary.shape)
print("Candidate tract pool 2024 shape:", tract_candidate_pool_2024.shape)
print("Candidate states:", tract_candidate_pool_2024["display_state"].nunique())

Candidate tract summary shape: (20659, 46)
Candidate tract pool 2024 shape: (20659, 245)
Candidate states: 51


### Cell 17 — load 2024 tract geometry for candidate states

In [52]:
# Cell 17 — load 2024 tract geometry for candidate states

needed_state_fips = sorted(
    tract_candidate_pool_2024["geoid"].astype(str).str[:2].dropna().unique().tolist()
)

print("Needed state FIPS:", needed_state_fips)

tract_gdf_list = []

for st in needed_state_fips:
    zip_name = f"tl_2024_{st}_tract.zip"
    shp_name = f"tl_2024_{st}_tract.shp"

    zip_path = TRACT_ROOT_DIR / zip_name
    extract_dir = TRACT_ROOT_DIR / f"tl_2024_{st}_tract"
    shp_path = extract_dir / shp_name

    if not shp_path.exists():
        extract_dir.mkdir(parents=True, exist_ok=True)

        if not zip_path.exists():
            url = f"{TRACT_BASE_URL}{zip_name}"
            print(f"Downloading {url} ...")
            r = requests.get(url, timeout=120)
            r.raise_for_status()
            with open(zip_path, "wb") as f:
                f.write(r.content)

        print(f"Extracting {zip_name} ...")
        with zipfile.ZipFile(zip_path, "r") as zf:
            zf.extractall(extract_dir)

    if not shp_path.exists():
        raise FileNotFoundError(f"Missing shapefile after extraction: {shp_path}")

    gdf_state = gpd.read_file(shp_path)
    tract_gdf_list.append(gdf_state)

tract_gdf = pd.concat(tract_gdf_list, ignore_index=True)
tract_gdf = gpd.GeoDataFrame(tract_gdf, geometry="geometry", crs=tract_gdf_list[0].crs)

tract_gdf["geoid"] = normalize_geoid_series(tract_gdf["GEOID"])
tract_gdf = tract_gdf.dropna(subset=["geoid", "geometry"]).copy()
tract_gdf = tract_gdf[tract_gdf.geometry.notna()].copy()
tract_gdf = tract_gdf[~tract_gdf.geometry.is_empty].copy()
tract_gdf = tract_gdf.drop_duplicates(subset=["geoid"]).copy()

print("tract_gdf shape:", tract_gdf.shape)
print("tract_gdf unique GEOIDs:", tract_gdf["geoid"].nunique())

Needed state FIPS: ['01', '02', '04', '05', '06', '08', '10', '11', '12', '13', '15', '16', '17', '18', '19', '20', '21', '22', '23', '24', '25', '26', '27', '28', '29', '30', '31', '32', '33', '34', '35', '36', '37', '38', '39', '40', '41', '42', '44', '45', '46', '47', '48', '49', '50', '51', '53', '54', '55', '56', '72']
Extracting tl_2024_02_tract.zip ...
Extracting tl_2024_06_tract.zip ...
Extracting tl_2024_08_tract.zip ...
Extracting tl_2024_10_tract.zip ...
Extracting tl_2024_11_tract.zip ...
Extracting tl_2024_15_tract.zip ...
Extracting tl_2024_16_tract.zip ...
Extracting tl_2024_19_tract.zip ...
Extracting tl_2024_20_tract.zip ...
Extracting tl_2024_21_tract.zip ...
Extracting tl_2024_23_tract.zip ...
Extracting tl_2024_25_tract.zip ...
Extracting tl_2024_27_tract.zip ...
Extracting tl_2024_30_tract.zip ...
Extracting tl_2024_31_tract.zip ...
Extracting tl_2024_33_tract.zip ...
Extracting tl_2024_34_tract.zip ...
Extracting tl_2024_36_tract.zip ...
Extracting tl_2024_38_trac

### Cell 18 — adjacency graph from candidate tracts

In [53]:
# Cell 18 — adjacency graph from candidate tracts

candidate_gdf = tract_gdf.merge(
    tract_candidate_pool_2024,
    on="geoid",
    how="inner",
    validate="1:1"
)

candidate_gdf = gpd.GeoDataFrame(candidate_gdf, geometry="geometry", crs=tract_gdf.crs)

print("Candidate geometry shape:", candidate_gdf.shape)
print("Candidate tracts with geometry:", candidate_gdf["geoid"].nunique())

adjacency_edges = build_touch_edges(candidate_gdf)

print("Adjacency edges shape:", adjacency_edges.shape)
print(adjacency_edges.head(20))

Candidate geometry shape: (20659, 259)
Candidate tracts with geometry: 20659
Adjacency edges shape: (30074, 4)
   display_state  display_county      geoid_1      geoid_2
0        Alabama  Autauga County  01001021000  01001021100
1        Alabama  Baldwin County  01003011603  01003011604
2        Alabama  Baldwin County  01003010500  01003010600
3        Alabama  Barbour County  01005950100  01005950900
4        Alabama  Barbour County  01005950100  01005950200
5        Alabama  Barbour County  01005950100  01005950800
6        Alabama  Barbour County  01005950100  01005950700
7        Alabama  Barbour County  01005950200  01005950500
8        Alabama  Barbour County  01005950200  01005950300
9        Alabama  Barbour County  01005950200  01005950700
10       Alabama  Barbour County  01005950300  01005950400
11       Alabama  Barbour County  01005950300  01005950500
12       Alabama  Barbour County  01005950500  01005950900
13       Alabama  Barbour County  01005950400  01005950500
14  

### Cell 19 — connected clusters from eligible 2024 tracts

In [54]:
# Cell 19 — connected clusters from eligible 2024 tracts

all_candidate_nodes = candidate_gdf["geoid"].astype(str).unique().tolist()
node_to_cluster = connected_components_from_edges(all_candidate_nodes, adjacency_edges)

candidate_gdf["cluster_id_num"] = candidate_gdf["geoid"].astype(str).map(node_to_cluster)
candidate_gdf["cluster_id"] = (
    candidate_gdf["display_state"].astype(str) + " | " +
    candidate_gdf["display_county"].astype(str) + " | cluster_" +
    candidate_gdf["cluster_id_num"].astype(str)
)

cluster_members_2024 = candidate_gdf.copy()

cluster_static_2024 = (
    cluster_members_2024
    .groupby(["cluster_id", "display_state", "display_county"], dropna=False)
    .agg(
        n_cluster_tracts=("geoid", "nunique"),
        cluster_total_pop_2024=("pop_total", "sum"),
        mean_igs_total_2024=("igs_total", "mean"),
        mean_igs_economy_2024=("igs_economy", "mean"),
        tiny_tract_share_2024=("tiny_tract_flag", "mean"),
        zero_igs_share_2024=("zero_igs_flag", "mean"),
        member_geoids=("geoid", lambda s: ", ".join(s.astype(str).tolist())),
        member_names=("display_name", lambda s: " | ".join(s.astype(str).tolist())),
    )
    .reset_index()
)

cluster_static_2024 = cluster_static_2024.sort_values(
    ["n_cluster_tracts", "cluster_total_pop_2024", "mean_igs_total_2024"],
    ascending=[False, False, True]
).reset_index(drop=True)

serious_clusters_base = cluster_static_2024.loc[
    (cluster_static_2024["n_cluster_tracts"] >= MIN_CLUSTER_TRACTS) &
    (cluster_static_2024["cluster_total_pop_2024"] >= MIN_CLUSTER_POP)
].copy()

print("All clusters:", cluster_static_2024.shape)
print("Serious clusters base:", serious_clusters_base.shape)
print()
print(serious_clusters_base.head(25))

All clusters: (4808, 11)
Serious clusters base: (1662, 11)

                                           cluster_id display_state       display_county  n_cluster_tracts  cluster_total_pop_2024  mean_igs_total_2024  mean_igs_economy_2024  tiny_tract_share_2024  \
0               Illinois | Cook County | cluster_1349      Illinois          Cook County               392               1194021.0            35.661480              33.618112               0.160714   
1       California | Los Angeles County | cluster_311    California   Los Angeles County               373               1400932.0            36.593298              38.603485               0.010724   
2              Michigan | Wayne County | cluster_2108      Michigan         Wayne County               273                670747.0            35.524632              32.307326               0.208791   
3              New York | Bronx County | cluster_2859      New York         Bronx County               259               1082614.0      

### Cell 19.5 — broader adjacency scope

In [57]:
# Cell 19.5 — broader adjacency scope

SEED_COUNTIES_N = 15
BROADER_SCORE_FLOOR = 80.0
BROADER_MAX_PER_COUNTY = 20

# take the top counties that already surfaced in candidate_communities
seed_counties = (
    candidate_communities
    .head(SEED_COUNTIES_N)[["display_state", "display_county"]]
    .drop_duplicates()
    .copy()
)

# pull in more shortlisted tracts from those counties, not just the top-150 national rows
broader_county_pool = (
    tract_shortlist_scored
    .merge(seed_counties, on=["display_state", "display_county"], how="inner")
    .loc[lambda d: d["total_shortlist_score"] >= BROADER_SCORE_FLOOR]
    .copy()
)

# keep the strongest tracts per county so adjacency stays computationally manageable
broader_county_pool["county_rank_by_score"] = (
    broader_county_pool
    .groupby(["display_state", "display_county"])["total_shortlist_score"]
    .rank(method="first", ascending=False)
)

broader_county_pool = (
    broader_county_pool
    .loc[broader_county_pool["county_rank_by_score"] <= BROADER_MAX_PER_COUNTY]
    .drop(columns=["county_rank_by_score"])
    .copy()
)

# build the broader adjacency scope
CLUSTER_SCOPE_DF = (
    pd.concat(
        [
            top_national_candidates,
            broader_county_pool,
        ],
        ignore_index=True
    )
    .drop_duplicates(subset=["geoid"])
    .sort_values(
        ["display_state", "display_county", "total_shortlist_score"],
        ascending=[True, True, False]
    )
    .reset_index(drop=True)
)

print("Seed counties:", len(seed_counties))
print(seed_counties)

print()
print("Broader county pool shape:", broader_county_pool.shape)
print("Broader adjacency scope shape:", CLUSTER_SCOPE_DF.shape)
print("Unique counties in broader scope:",
      CLUSTER_SCOPE_DF[["display_state", "display_county"]].drop_duplicates().shape[0])

print()
print("Top counties by tract count in broader scope:")
print(
    CLUSTER_SCOPE_DF
    .assign(state_county=lambda d: d["display_county"].astype(str) + ", " + d["display_state"].astype(str))
    ["state_county"]
    .value_counts()
    .head(20)
)

Seed counties: 15
   display_state      display_county
0    Puerto Rico  San Juan Municipio
1       Illinois         Cook County
2        Alabama       Mobile County
3      Tennessee       Shelby County
4           Ohio     Cuyahoga County
5       Michigan        Wayne County
6      Louisiana      Orleans Parish
7        Georgia         Bibb County
8        Indiana         Lake County
9    Puerto Rico   Arecibo Municipio
10       Arizona       Navajo County
11      Missouri      St. Louis city
12       Arizona       Apache County
13   Puerto Rico   Guánica Municipio
14       Florida       Orange County

Broader county pool shape: (143, 33)
Broader adjacency scope shape: (213, 33)
Unique counties in broader scope: 80

Top counties by tract count in broader scope:
state_county
San Juan Municipio, Puerto Rico      20
Cook County, Illinois                20
Wayne County, Michigan               16
Shelby County, Tennessee             14
Cuyahoga County, Ohio                11
Lake County, I

### Cell 20 — build the cluster-year panel

In [60]:
# Cell 20 — cluster-year panel

cluster_lookup = cluster_members_2024[["cluster_id", "display_state", "display_county", "geoid"]].drop_duplicates().copy()

cluster_panel_source = model_df.merge(
    cluster_lookup,
    on="geoid",
    how="inner",
    validate="m:1"
).copy()

cluster_year_metrics = [
    "igs_total",
    "igs_economy",
    "poverty_rate",
    "unemp_rate",
    "median_household_income",
    "lfpr_16p",
]

def summarize_cluster_year(grp: pd.DataFrame) -> pd.Series:
    w = grp["pop_total"]
    return pd.Series({
        "cluster_total_pop": grp["pop_total"].sum(skipna=True),
        "n_cluster_tracts_present": grp["geoid"].nunique(),
        "igs_total": weighted_mean(grp["igs_total"], w),
        "igs_economy": weighted_mean(grp["igs_economy"], w),
        "poverty_rate": weighted_mean(grp["poverty_rate"], w),
        "unemp_rate": weighted_mean(grp["unemp_rate"], w),
        "median_household_income": weighted_mean(grp["median_household_income"], w),
        "lfpr_16p": weighted_mean(grp["lfpr_16p"], w),
    })

cluster_year_panel = (
    cluster_panel_source
    .groupby(["cluster_id", "display_state", "display_county", "year"], dropna=False)
    .apply(summarize_cluster_year)
    .reset_index()
)

print("cluster_year_panel shape:", cluster_year_panel.shape)
print(cluster_year_panel.head(20))

KeyError: 'display_state'

### Cell 21 — cluster-level period summaries and flags

In [ ]:
# Cell 21 — cluster-level period summaries and flags

cluster_period_cols = [
    "igs_total",
    "igs_economy",
    "poverty_rate",
    "unemp_rate",
    "median_household_income",
    "lfpr_16p",
    "cluster_total_pop",
]

cluster_pre = add_cluster_period_means(cluster_year_panel, cluster_period_cols, PRE_YEARS, "pre")
cluster_shock = add_cluster_period_means(cluster_year_panel, cluster_period_cols, SHOCK_YEARS, "shock")
cluster_recovery = add_cluster_period_means(cluster_year_panel, cluster_period_cols, RECOVERY_YEARS, "recovery")

cluster_2024 = (
    cluster_year_panel.loc[cluster_year_panel["year"] == CURRENT_YEAR, ["cluster_id"] + cluster_period_cols]
    .groupby("cluster_id", as_index=False)[cluster_period_cols]
    .mean()
    .rename(columns={c: f"y2024_{c}" for c in cluster_period_cols})
)

cluster_year_counts = (
    cluster_year_panel.groupby("cluster_id", as_index=False)
    .agg(
        cluster_years_observed=("year", "nunique"),
        years_cluster_below_45=("igs_total", lambda s: int((s < LOW_IGS_THRESHOLD).sum())),
    )
)

cluster_summary = (
    serious_clusters_base
    .merge(cluster_pre, on="cluster_id", how="left")
    .merge(cluster_shock, on="cluster_id", how="left")
    .merge(cluster_recovery, on="cluster_id", how="left")
    .merge(cluster_2024, on="cluster_id", how="left")
    .merge(cluster_year_counts, on="cluster_id", how="left")
    .copy()
)

# trajectory metrics
cluster_summary["igs_total_shock_drop"] = cluster_summary["shock_igs_total"] - cluster_summary["pre_igs_total"]
cluster_summary["igs_total_recovery_gap"] = cluster_summary["recovery_igs_total"] - cluster_summary["pre_igs_total"]
cluster_summary["igs_economy_shock_drop"] = cluster_summary["shock_igs_economy"] - cluster_summary["pre_igs_economy"]
cluster_summary["igs_economy_recovery_gap"] = cluster_summary["recovery_igs_economy"] - cluster_summary["pre_igs_economy"]

# cluster-level econ vulnerability using recovery-period values
cq_igs_econ_low = cluster_summary["recovery_igs_economy"].quantile(ECON_VULN_PCTL)
cq_poverty_high = cluster_summary["recovery_poverty_rate"].quantile(1 - ECON_VULN_PCTL)
cq_unemp_high = cluster_summary["recovery_unemp_rate"].quantile(1 - ECON_VULN_PCTL)
cq_income_low = cluster_summary["recovery_median_household_income"].quantile(ECON_VULN_PCTL)
cq_lfpr_low = cluster_summary["recovery_lfpr_16p"].quantile(ECON_VULN_PCTL)

cluster_summary["econ_cond_low_igs_economy"] = cluster_summary["recovery_igs_economy"] <= cq_igs_econ_low
cluster_summary["econ_cond_high_poverty"] = cluster_summary["recovery_poverty_rate"] >= cq_poverty_high
cluster_summary["econ_cond_high_unemp"] = cluster_summary["recovery_unemp_rate"] >= cq_unemp_high
cluster_summary["econ_cond_low_income"] = cluster_summary["recovery_median_household_income"] <= cq_income_low
cluster_summary["econ_cond_low_lfpr"] = cluster_summary["recovery_lfpr_16p"] <= cq_lfpr_low

cluster_econ_cond_cols = [
    "econ_cond_low_igs_economy",
    "econ_cond_high_poverty",
    "econ_cond_high_unemp",
    "econ_cond_low_income",
    "econ_cond_low_lfpr",
]

cluster_summary["econ_vuln_cond_count"] = cluster_summary[cluster_econ_cond_cols].sum(axis=1)
cluster_summary["econ_vulnerable_cluster_flag"] = cluster_summary["econ_vuln_cond_count"] >= ECON_VULN_MIN_CONDS

# challenge/story flags
cluster_summary["pre_covid_solid_flag"] = cluster_summary["pre_igs_total"] >= LOW_IGS_THRESHOLD
cluster_summary["still_low_flag"] = (
    (cluster_summary["y2024_igs_total"] < LOW_IGS_THRESHOLD) |
    (cluster_summary["recovery_igs_total"] < LOW_IGS_THRESHOLD)
)
cluster_summary["weak_recovery_flag"] = (
    (cluster_summary["igs_total_recovery_gap"] <= -3) |
    (cluster_summary["igs_economy_recovery_gap"] <= -3)
)

print("cluster_summary shape:", cluster_summary.shape)
print(cluster_summary.head(20))

### Cell 22 — trajectory-first cluster score

In [61]:
# Cell 22 — trajectory-first cluster score

trajectory_ranked_clusters = cluster_summary.copy()

# A. Trajectory / weak recovery — 35 pts
trajectory_ranked_clusters["score_pre_covid_baseline"] = percentile_score(
    trajectory_ranked_clusters["pre_igs_total"], 10, higher_is_better=True
)
trajectory_ranked_clusters["score_total_shock_drop"] = percentile_score(
    -trajectory_ranked_clusters["igs_total_shock_drop"], 10, higher_is_better=True
)
trajectory_ranked_clusters["score_total_recovery_gap"] = percentile_score(
    -trajectory_ranked_clusters["igs_total_recovery_gap"], 8, higher_is_better=True
)
trajectory_ranked_clusters["score_econ_recovery_gap"] = percentile_score(
    -trajectory_ranked_clusters["igs_economy_recovery_gap"], 7, higher_is_better=True
)
trajectory_ranked_clusters["trajectory_block_score"] = (
    trajectory_ranked_clusters["score_pre_covid_baseline"] +
    trajectory_ranked_clusters["score_total_shock_drop"] +
    trajectory_ranked_clusters["score_total_recovery_gap"] +
    trajectory_ranked_clusters["score_econ_recovery_gap"]
)

# B. Persistent low IGS — 25 pts
trajectory_ranked_clusters["score_years_below_45"] = percentile_score(
    trajectory_ranked_clusters["years_cluster_below_45"], 10, higher_is_better=True
)
trajectory_ranked_clusters["score_y2024_igs_total"] = percentile_score(
    trajectory_ranked_clusters["y2024_igs_total"], 8, higher_is_better=False
)
trajectory_ranked_clusters["score_y2024_igs_economy"] = percentile_score(
    trajectory_ranked_clusters["y2024_igs_economy"], 7, higher_is_better=False
)
trajectory_ranked_clusters["persistent_low_block_score"] = (
    trajectory_ranked_clusters["score_years_below_45"] +
    trajectory_ranked_clusters["score_y2024_igs_total"] +
    trajectory_ranked_clusters["score_y2024_igs_economy"]
)

# C. Economic vulnerability over time — 20 pts
trajectory_ranked_clusters["score_recovery_poverty"] = percentile_score(
    trajectory_ranked_clusters["recovery_poverty_rate"], 6, higher_is_better=True
)
trajectory_ranked_clusters["score_recovery_unemp"] = percentile_score(
    trajectory_ranked_clusters["recovery_unemp_rate"], 5, higher_is_better=True
)
trajectory_ranked_clusters["score_recovery_income"] = percentile_score(
    trajectory_ranked_clusters["recovery_median_household_income"], 5, higher_is_better=False
)
trajectory_ranked_clusters["score_recovery_lfpr"] = percentile_score(
    trajectory_ranked_clusters["recovery_lfpr_16p"], 4, higher_is_better=False
)
trajectory_ranked_clusters["econ_vulnerability_block_score"] = (
    trajectory_ranked_clusters["score_recovery_poverty"] +
    trajectory_ranked_clusters["score_recovery_unemp"] +
    trajectory_ranked_clusters["score_recovery_income"] +
    trajectory_ranked_clusters["score_recovery_lfpr"]
)

# D. Cluster coherence / study value — 10 pts
trajectory_ranked_clusters["score_n_cluster_tracts"] = percentile_score(
    trajectory_ranked_clusters["n_cluster_tracts"], 4, higher_is_better=True
)
trajectory_ranked_clusters["score_cluster_pop"] = percentile_score(
    trajectory_ranked_clusters["cluster_total_pop_2024"], 3, higher_is_better=True
)
trajectory_ranked_clusters["score_years_observed"] = percentile_score(
    trajectory_ranked_clusters["cluster_years_observed"], 3, higher_is_better=True
)
trajectory_ranked_clusters["coherence_block_score"] = (
    trajectory_ranked_clusters["score_n_cluster_tracts"] +
    trajectory_ranked_clusters["score_cluster_pop"] +
    trajectory_ranked_clusters["score_years_observed"]
)

# E. Data quality / non-artifact — 10 pts
trajectory_ranked_clusters["score_zero_igs_quality"] = percentile_score(
    1 - trajectory_ranked_clusters["zero_igs_share_2024"], 5, higher_is_better=True
)
trajectory_ranked_clusters["score_tiny_tract_quality"] = percentile_score(
    1 - trajectory_ranked_clusters["tiny_tract_share_2024"], 5, higher_is_better=True
)
trajectory_ranked_clusters["data_quality_block_score"] = (
    trajectory_ranked_clusters["score_zero_igs_quality"] +
    trajectory_ranked_clusters["score_tiny_tract_quality"]
)

trajectory_ranked_clusters["total_trajectory_score"] = (
    trajectory_ranked_clusters["trajectory_block_score"] +
    trajectory_ranked_clusters["persistent_low_block_score"] +
    trajectory_ranked_clusters["econ_vulnerability_block_score"] +
    trajectory_ranked_clusters["coherence_block_score"] +
    trajectory_ranked_clusters["data_quality_block_score"]
)

# final target flag: fits your new story idea
trajectory_ranked_clusters["trajectory_target_flag"] = (
    trajectory_ranked_clusters["pre_covid_solid_flag"] &
    trajectory_ranked_clusters["still_low_flag"] &
    trajectory_ranked_clusters["weak_recovery_flag"] &
    trajectory_ranked_clusters["econ_vulnerable_cluster_flag"] &
    (trajectory_ranked_clusters["tiny_tract_share_2024"] <= MAX_TINY_TRACT_SHARE) &
    (trajectory_ranked_clusters["zero_igs_share_2024"] <= MAX_ZERO_IGS_SHARE)
)

trajectory_ranked_clusters = trajectory_ranked_clusters.sort_values(
    ["trajectory_target_flag", "total_trajectory_score", "y2024_igs_total"],
    ascending=[False, False, True]
).reset_index(drop=True)

trajectory_target_clusters = trajectory_ranked_clusters.loc[
    trajectory_ranked_clusters["trajectory_target_flag"]
].copy().reset_index(drop=True)

print("Ranked clusters:", trajectory_ranked_clusters.shape)
print("Trajectory target clusters:", trajectory_target_clusters.shape)
print()
print(trajectory_ranked_clusters[
    [
        "cluster_id", "display_state", "display_county",
        "n_cluster_tracts", "cluster_total_pop_2024",
        "pre_igs_total", "recovery_igs_total", "y2024_igs_total",
        "pre_igs_economy", "recovery_igs_economy", "y2024_igs_economy",
        "econ_vuln_cond_count",
        "pre_covid_solid_flag", "weak_recovery_flag", "still_low_flag", "trajectory_target_flag",
        "total_trajectory_score"
    ]
].head(25))

NameError: name 'cluster_summary' is not defined

### Cell 23 — clean outputs for next-step enrichment

In [62]:
# Cell 23 — outputs for next-step healthcare/business enrichment

# one row per tract in each serious / target cluster
cluster_tract_lookup = cluster_members_2024[
    [
        "cluster_id", "display_state", "display_county", "display_name",
        "geoid", "state", "county", "tract",
        "igs_total", "igs_economy", "pop_total"
    ]
].copy()

cluster_tract_lookup["state_fips"] = cluster_tract_lookup["geoid"].str[:2]
cluster_tract_lookup["county_fips"] = cluster_tract_lookup["geoid"].str[2:5]
cluster_tract_lookup["tract_code"] = cluster_tract_lookup["geoid"].str[5:]

# cluster-level shortlist for enrichment
cluster_enrichment_queue = trajectory_ranked_clusters[
    [
        "cluster_id", "display_state", "display_county",
        "n_cluster_tracts", "cluster_total_pop_2024",
        "member_geoids", "member_names",
        "pre_igs_total", "recovery_igs_total", "y2024_igs_total",
        "pre_igs_economy", "recovery_igs_economy", "y2024_igs_economy",
        "recovery_poverty_rate", "recovery_unemp_rate", "recovery_median_household_income", "recovery_lfpr_16p",
        "years_cluster_below_45",
        "tiny_tract_share_2024", "zero_igs_share_2024",
        "econ_vuln_cond_count",
        "pre_covid_solid_flag", "weak_recovery_flag", "still_low_flag", "trajectory_target_flag",
        "total_trajectory_score"
    ]
].copy()

if SAVE_INTERMEDIATE:
    safe_save_parquet(tract_candidate_summary, OUT_DIR / "tract_candidate_summary.parquet")
    safe_save_parquet(tract_candidate_pool_2024, OUT_DIR / "tract_candidate_pool_2024.parquet")
    safe_save_parquet(cluster_members_2024.drop(columns="geometry"), OUT_DIR / "cluster_members_2024.parquet")
    safe_save_parquet(cluster_year_panel, OUT_DIR / "cluster_year_panel.parquet")
    safe_save_parquet(cluster_summary, OUT_DIR / "cluster_summary.parquet")
    safe_save_parquet(trajectory_ranked_clusters, OUT_DIR / "trajectory_ranked_clusters.parquet")
    safe_save_parquet(trajectory_target_clusters, OUT_DIR / "trajectory_target_clusters.parquet")
    safe_save_parquet(cluster_tract_lookup, OUT_DIR / "cluster_tract_lookup.parquet")

tract_candidate_summary.to_csv(OUT_DIR / "tract_candidate_summary.csv", index=False)
tract_candidate_pool_2024.to_csv(OUT_DIR / "tract_candidate_pool_2024.csv", index=False)
cluster_members_2024.drop(columns="geometry").to_csv(OUT_DIR / "cluster_members_2024.csv", index=False)
cluster_year_panel.to_csv(OUT_DIR / "cluster_year_panel.csv", index=False)
cluster_summary.to_csv(OUT_DIR / "cluster_summary.csv", index=False)
trajectory_ranked_clusters.to_csv(OUT_DIR / "trajectory_ranked_clusters.csv", index=False)
trajectory_target_clusters.to_csv(OUT_DIR / "trajectory_target_clusters.csv", index=False)
cluster_tract_lookup.to_csv(OUT_DIR / "cluster_tract_lookup.csv", index=False)

print("Saved outputs to:", OUT_DIR)

NameError: name 'trajectory_ranked_clusters' is not defined

### Cell 24 — quick review tables

In [63]:
# Cell 24 — quick review tables

print("Top trajectory-ranked clusters:")
print(
    trajectory_ranked_clusters[
        [
            "cluster_id", "display_state", "display_county",
            "n_cluster_tracts", "cluster_total_pop_2024",
            "pre_igs_total", "shock_igs_total", "recovery_igs_total", "y2024_igs_total",
            "pre_igs_economy", "shock_igs_economy", "recovery_igs_economy", "y2024_igs_economy",
            "years_cluster_below_45", "econ_vuln_cond_count",
            "tiny_tract_share_2024", "zero_igs_share_2024",
            "trajectory_target_flag", "total_trajectory_score"
        ]
    ].head(25)
)

print()
print("Trajectory target clusters:")
print(
    trajectory_target_clusters[
        [
            "cluster_id", "display_state", "display_county",
            "n_cluster_tracts", "cluster_total_pop_2024",
            "member_geoids",
            "pre_igs_total", "recovery_igs_total", "y2024_igs_total",
            "pre_igs_economy", "recovery_igs_economy", "y2024_igs_economy",
            "recovery_poverty_rate", "recovery_unemp_rate",
            "recovery_median_household_income", "recovery_lfpr_16p",
            "total_trajectory_score"
        ]
    ].head(25)
)

Top trajectory-ranked clusters:


NameError: name 'trajectory_ranked_clusters' is not defined

### Cell 25 — cluster summaries + serious cluster filter

In [64]:
# Cell 25 — cluster summaries + serious cluster filter

adjacent_cluster_summary = (
    adjacency_gdf
    .groupby(["cluster_id", "display_state", "display_county"], dropna=False)
    .agg(
        n_cluster_tracts=("geoid", "nunique"),
        cluster_total_pop=("pop_total", "sum"),
        mean_shortlist_score=("total_shortlist_score", "mean"),
        max_shortlist_score=("total_shortlist_score", "max"),
        mean_igs_total=("igs_total", "mean"),
        min_igs_total=("igs_total", "min"),
        mean_poverty_rate=("poverty_rate", "mean"),
        mean_insured_share=("insured_share", "mean"),
        mean_unemp_rate=("unemp_rate", "mean"),
        tiny_tract_share=("tiny_tract_flag", "mean"),
        member_geoids=("geoid", lambda s: ", ".join(s.astype(str).tolist())),
        member_names=("display_name", lambda s: " | ".join(s.astype(str).tolist())),
    )
    .reset_index()
    .sort_values(
        ["n_cluster_tracts", "mean_shortlist_score", "cluster_total_pop"],
        ascending=[False, False, False]
    )
    .reset_index(drop=True)
)

serious_adjacent_clusters = (
    adjacent_cluster_summary[
        (adjacent_cluster_summary["n_cluster_tracts"] >= MIN_CLUSTER_TRACTS) &
        (adjacent_cluster_summary["cluster_total_pop"] >= MIN_CLUSTER_POP)
    ]
    .copy()
    .reset_index(drop=True)
)

print("All adjacent clusters:", adjacent_cluster_summary.shape)
print("Serious adjacent clusters:", serious_adjacent_clusters.shape)
print()

print("Top adjacent clusters:")
print(adjacent_cluster_summary.head(25))
print()

print("Top serious adjacent clusters:")
print(serious_adjacent_clusters.head(25))

All adjacent clusters: (139, 15)
Serious adjacent clusters: (14, 15)

Top adjacent clusters:
                                        cluster_id  display_state      display_county  n_cluster_tracts  cluster_total_pop  mean_shortlist_score  max_shortlist_score  mean_igs_total  min_igs_total  \
0   Puerto Rico | San Juan Municipio | cluster_126    Puerto Rico  San Juan Municipio                10            20465.0             85.477542            91.089515       25.910000           22.0   
1               Indiana | Lake County | cluster_42        Indiana         Lake County                 7             8138.0             84.870445            88.707871       24.142857           21.0   
2             Arizona | Apache County | cluster_13        Arizona       Apache County                 7            24205.0             83.667499            88.431911       21.000000            0.0   
3              Illinois | Cook County | cluster_35       Illinois         Cook County                 6    

### Optional Cell 26 — save adjacency outputs

In [65]:
# Cell 26 — save adjacency outputs

if SAVE_INTERMEDIATE:
    safe_save_parquet(adjacency_gdf.drop(columns="geometry"), OUT_DIR / "adjacency_cluster_members.parquet")
    safe_save_parquet(adjacent_cluster_summary, OUT_DIR / "adjacent_cluster_summary.parquet")
    safe_save_parquet(serious_adjacent_clusters, OUT_DIR / "serious_adjacent_clusters.parquet")

adjacency_gdf.drop(columns="geometry").to_csv(OUT_DIR / "adjacency_cluster_members.csv", index=False)
adjacent_cluster_summary.to_csv(OUT_DIR / "adjacent_cluster_summary.csv", index=False)
serious_adjacent_clusters.to_csv(OUT_DIR / "serious_adjacent_clusters.csv", index=False)

print("Saved adjacency outputs to:", OUT_DIR)

Saved adjacency outputs to: data\processed


In [66]:
# Cell 25.6 — one row per tract for each serious cluster

serious_cluster_tracts = (
    adjacency_gdf.loc[
        adjacency_gdf["cluster_id"].isin(serious_adjacent_clusters["cluster_id"])
    ,
        [
            "cluster_id",
            "display_state",
            "display_county",
            "display_name",
            "geoid",
            "igs_total",
            "total_shortlist_score"
        ]
    ]
    .sort_values(["cluster_id", "geoid"])
    .reset_index(drop=True)
    .copy()
)

# split the 11-digit tract code into parts in case the website asks for them separately
serious_cluster_tracts["state_fips"] = serious_cluster_tracts["geoid"].str[:2]
serious_cluster_tracts["county_fips"] = serious_cluster_tracts["geoid"].str[2:5]
serious_cluster_tracts["tract_code"] = serious_cluster_tracts["geoid"].str[5:]

print(serious_cluster_tracts.to_string(index=False))

                                    cluster_id display_state     display_county                                        display_name       geoid  igs_total  total_shortlist_score state_fips county_fips tract_code
          Arizona | Apache County | cluster_13       Arizona      Apache County           Census Tract 9426; Apache County; Arizona 04001942600        0.0              80.689922         04         001     942600
          Arizona | Apache County | cluster_13       Arizona      Apache County           Census Tract 9441; Apache County; Arizona 04001944100       26.0              84.260294         04         001     944100
          Arizona | Apache County | cluster_13       Arizona      Apache County        Census Tract 9443.01; Apache County; Arizona 04001944301       21.0              88.431911         04         001     944301
          Arizona | Apache County | cluster_13       Arizona      Apache County        Census Tract 9443.02; Apache County; Arizona 04001944302       21